[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/odometry/odometry.ipynb)

# The Most Likely Trajectory from Odometry

A robot drives a lap and reads, at every stop, how far it moved since the last one. Each reading is a little off, so adding them up, dead reckoning, drifts. Back at the start it recognises it, reads one more relative pose, and the lap closes. This notebook finds the poses most likely given all the readings, and how uncertain each one is. Every reading pulls only on the two poses it links, and nothing between every two poses is ever held.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, concatenate
from numga.algebras import PGA2D
from examples.animation import save_animation
from examples.geometry.odometry import render

np.set_printoptions(precision=4, suppress=True)

ga = PGA2D
mv = NumpyContext(ga).multivector
# A pose, and a small motion on the right of a pose.
Motor = ga.gatype.rotor()
Twist = ga.gatype.bivector()
# A line reads out a twist: `line & twist`.
Line = ga.gatype.antibivector()
# How uncertain a twist is: from a line, the twist it expects along it. Its inverse is a quadric on twists.
Covariance = ga.gatype((Twist, Line))
# The inverse of a covariance: a quadric on twists, `twist & information(twist)`.
Information = ga.gatype((Line, Twist))

## 1. The lap and its readings

The robot drives one lap in 36 steps of 0.7 m, turning faster and slower twice a lap, slipping sideways a little, one way and back twice a lap. Each reading is a relative pose, the pose at its head as read from the pose at its tail, with an error: a twist on the right, drawn from the reading's covariance, a map from lines to twists. The readings are every step, read to 3 cm and 0.01 rad, then one more that closes the loop: the last pose read from the first, recognising the start, to a millimetre. Every pose has a prior: the first is known, the others only vaguely.

In [ ]:
STEPS = 36
POSES = STEPS + 1


def isotropic(translation_std: float, rotation_std: float) -> Covariance:
    """The covariance of a reading with isotropic translation noise and rotation noise about the head."""
    translation = mv.yw * (mv.yw & Line) + mv.wx * (mv.wx & Line)               # [] Covariance
    rotation = mv.xy * (mv.xy & Line)                                           # [] Covariance
    return translation * translation_std**2 + rotation * rotation_std**2


def modes(uncertainty: Covariance) -> Twist:
    """Twists whose outer products sum to each uncertainty: along lines orthonormal in its form."""
    spread = Line & uncertainty                                                 # [...] Scalar <- (Line, Line)
    _, lines = spread.eigh(spread)                                              # [..., modes] Line
    return uncertainty[..., None](lines)                                        # [..., modes] Twist


rng = np.random.default_rng(9)
phase = 2 * np.pi * np.arange(STEPS) / STEPS
turn = mv.xy * (1 + 0.4 * np.sin(2 * phase)) * 2 * np.pi / STEPS                 # [steps] Twist
slip = mv.yw * 0.1 * np.cos(2 * phase)                                           # [steps] Twist
steps = ((turn + slip + mv.xw * 0.7) * 0.5).exp()                                # [steps] Motor
# Each step composes on the right, in the robot's own frame: a running product in that order is the
# reverse of the running product of the reverses.
truth = concatenate([mv.rotor()[None], steps.reverse().cumprod(axis=0).reverse()])                # [poses] Motor

# The pose each reading leaves from and arrives at: every step, then the last pose from the first.
tails = np.append(np.arange(STEPS), 0)                                            # [readings]
heads = np.append(np.arange(1, POSES), STEPS)                                    # [readings]
noises = concatenate([isotropic(0.03, 0.01) * np.ones(STEPS), isotropic(0.001, 0.0003)[None]])   # [readings] Covariance
spread = modes(noises)                                                           # [readings, modes] Twist
errors = (spread * rng.normal(size=spread.shape)).sum(axis=-1)                   # [readings] Twist
readings = (truth[tails].inverse() * truth[heads]) * (errors * 0.5).exp()        # [readings] Motor
# Every pose's prior: the first known, the others only vaguely.
priors = concatenate([isotropic(0.01, 0.01)[None], isotropic(100.0, 30.0) * np.ones(STEPS)])   # [poses] Covariance
# What each reading and each anchor weighs: the inverse of its covariance, a quadric on twists.
weights = noises.inverse()                                                       # [readings] Line <- Twist
anchor_weights = priors.inverse()                                                # [poses] Line <- Twist

## 2. Dead reckoning

Dead reckoning chains the step readings from the known start, and drifts. A small error of a pose is a twist on its right, `pose * (twist * 0.5).exp()`, and its uncertainty a covariance, a map from lines to twists. A dead-reckoned pose's uncertainty is the first pose's prior and the noise of every step before it, each carried to the pose. Carried to the world frame by the pose it enters at, `pose >> uncertainty(pose << Line)`, they add up along the lap; read back at every pose, they are its uncertainty, drawn as the ellipse two standard deviations out. Every pose but the first is anchored, only vaguely, where dead reckoning puts it.

In [ ]:
# The step readings, chained from the known start.
dead = concatenate([truth[:1], truth[0] * readings[:-1].reverse().cumprod(axis=0).reverse()])   # [poses] Motor
# Every pose's anchor: the first where it is known, the others where dead reckoning puts them.
anchors = concatenate([truth[:1], dead[1:]])                                     # [poses] Motor
# What enters at each pose: the first pose's prior, and the noise of the step that reaches it.
entering = concatenate([priors[:1], noises[:STEPS]])                             # [poses] Covariance
summed = (dead >> entering(dead << Line)).cumsum(axis=0)                         # [poses] Covariance
reckoned = dead << summed(dead >> Line)                                          # [poses] Covariance
render.draw_lap(truth, dead, reckoned, dead[[tails[-1], heads[-1]]]);

## 3. The problem

A reading measures the twist at its head minus the twist at its tail carried there by the motor between them, `relative << twist`, and its mismatch is the log of the motor between what it read and the relative motor of the poses. Mismatches at the readings and at the anchors pull on the poses: each weighted by the inverse of its covariance, a reading's at its head, and carried to its tail with the opposite sign. The pull of the mismatches is the gradient; the pull of what a correction measures is the information applied to it, never assembled; and how sharply the objective curves as each pose moves alone is what each reading weighs at its two ends.

In the notation of nonlinear least squares the information reads as the normal matrix $J^\top W J$, applied without forming it.

In [ ]:
def mismatch(reading: Motor, relative: Motor) -> Twist:
    """How far a relative motor is from what was read: a twist on the right of the reading."""
    return (reading.inverse() * relative).log() * 2                            # [readings] or [poses] Twist


def pull(relative: Motor, at_readings: Twist, at_anchors: Twist) -> Line:
    """What mismatches at the readings and at the anchors pull on every pose: each weighted by the inverse of
    its covariance, a reading's at its head, and carried to its tail with the opposite sign; an anchor's at
    its pose."""
    weighted = weights(at_readings)                                            # [..., readings] Line
    anchored = anchor_weights(at_anchors)                                      # [..., poses] Line
    # A reading pulls its head, and its tail the opposite way, carried there by the motor between them.
    at_tails = -(relative >> weighted)                                         # [..., readings] Line
    pulled = anchored.at[..., heads].add(weighted)                             # [..., poses] Line
    return pulled.at[..., tails].add(at_tails)                                 # [..., poses] Line


def gradient(poses: Motor) -> Line:
    """The gradient, per pose, of half the squared mismatches of every reading and of every pose from its
    anchor, each measured by its covariance: their pull."""
    relative = poses[tails].inverse() * poses[heads]                          # [readings] Motor: the head, from the tail
    return pull(relative, mismatch(readings, relative), mismatch(anchors, poses))   # [poses] Line


def curvature(poses: Motor, twists: Twist) -> Line:
    """The information applied to a correction of every pose: the pull of what the readings and the anchors
    measure of it."""
    relative = poses[tails].inverse() * poses[heads]                          # [readings] Motor
    measured = twists[..., heads] - (relative << twists[..., tails])           # [..., readings] Twist
    return pull(relative, measured, twists)                                    # [..., poses] Line


def curvature_alone(poses: Motor) -> Information:
    """How sharply the objective curves as each pose moves alone, every other pose held."""
    relative = poses[tails].inverse() * poses[heads]                          # [readings] Motor: the head, from the tail
    # A reading measures its head's twist as it is, and its tail's carried to the head: its head, moved
    # alone, meets the reading's weight, and its tail the weight carried back to it.
    at_tails = relative >> weights(relative << Twist)                          # [readings] Line <- Twist
    # Each pose's anchor, then every reading at its head and at its tail.
    alone = anchor_weights.at[heads].add(weights)                              # [poses] Line <- Twist
    return alone.at[tails].add(at_tails)                                       # [poses] Line <- Twist

## 4. The solvers

Conjugate gradients, preconditioned by each pose's curvature alone, solve the information for the correction a pull asks for, one solve for each leading index, ending exact after as many steps as there are unknowns. The motors make the problem nonlinear: Gauss-Newton solves at the current poses for the correction the gradient asks for, and moves the poses by a share of it, all of it for full steps.

In [ ]:
def conjugate_gradients(poses: Motor, right: Line):
    """The corrections the information sends to the given lines, one solve for each leading index,
    preconditioned by each pose's curvature alone. Yields the corrections after each iteration."""
    alone = curvature_alone(poses)                                             # [poses] Line <- Twist
    correction = 0 * alone.solve(right)                                        # [..., poses] Twist
    residual = right                                                           # [..., poses] Line
    direction = alone.solve(residual)                                          # [..., poses] Twist
    aligned = (residual & direction).sum(axis=-1)                              # [...] Scalar
    # Exact after as many steps as there are unknowns.
    for _ in range(len(Twist.output_subspace) * len(poses)):
        pushed = curvature(poses, direction)                                   # [..., poses] Line
        step = (aligned / (pushed & direction).sum(axis=-1))[..., None]        # [..., 1] Scalar
        correction = correction + direction * step                             # [..., poses] Twist
        residual = residual - pushed * step                                    # [..., poses] Line
        preconditioned = alone.solve(residual)                                 # [..., poses] Twist
        realigned = (residual & preconditioned).sum(axis=-1)                   # [...] Scalar
        direction = preconditioned + direction * (realigned / aligned)[..., None]   # [..., poses] Twist
        aligned = realigned
        yield correction


def gauss_newton(poses: Motor, damping: float, iterations: int):
    """Each iteration solves the information at the current poses for the correction the gradient asks for,
    and moves the poses by the damping's share of it. Yields the poses after each iteration."""
    for _ in range(iterations):
        *_, correction = conjugate_gradients(poses, -gradient(poses))   # [poses] Twist
        poses = poses * (correction * (0.5 * damping)).exp()                   # [poses] Motor
        yield poses

## 5. The most likely poses, and how uncertain each one is

A few full Gauss-Newton steps from dead reckoning reach the most likely poses. Their uncertainty never enters the solve: the information it solves is the whole uncertainty between the poses, held as its inverse. Each pose's own block of the inverse is its ellipse, and reading it out takes a solve of its own: what the information makes of every unit error a reading or an anchor allows, their outer products summed at each pose, all in one batch.

In [ ]:
def marginals(poses: Motor) -> Covariance:
    """Each pose's own uncertainty: the corrections the pull of every unit error a reading or an anchor
    allows asks for, their outer products summed at each pose."""
    relative = poses[tails].inverse() * poses[heads]                          # [readings] Motor
    sites = concatenate([modes(noises), modes(priors)])                        # [readings + poses, modes] Twist
    units = sites[..., None] * np.eye(len(sites))[:, None]                     # [readings + poses, modes, readings + poses] Twist
    right = pull(relative, units[..., :len(tails)], units[..., len(tails):])   # [readings + poses, modes, poses] Line
    *_, errors = conjugate_gradients(poses, right)                             # [readings + poses, modes, poses] Twist
    return (errors * (errors & Line)).sum(axis=(0, 1))                         # [poses] Covariance


*_, poses = gauss_newton(dead, 1.0, 5)                                           # [poses] Motor
uncertainty = marginals(poses)                                                   # [poses] Covariance
render.draw(truth, dead, reckoned, poses, uncertainty);

## 6. Closing the lap

Damped, Gauss-Newton moves the poses by only a share of each correction and relinearizes at the moved poses: each step closes the same share of what is left of their error, and so the square of that share of their uncertainty beyond what every reading allows at the step's poses. From dead reckoning, a fifth of the way at a time, the lap closes and every ellipse shrinks to what every reading allows.

In [ ]:
def uncertainties(iterates, uncertainty: Covariance, damping: float):
    """Each pose's uncertainty along damped Gauss-Newton, from the given one: each step's poses keep the rest
    of their error, so the rest's square of their uncertainty beyond what every reading allows there.
    Yields the poses of each step with it."""
    for poses in iterates:
        posterior = marginals(poses)                                           # [poses] Covariance
        uncertainty = posterior + (uncertainty - posterior) * (1 - damping)**2    # [poses] Covariance
        yield poses, uncertainty


damping = 0.2
closing = uncertainties(gauss_newton(dead, damping, 20), reckoned, damping)     # after each damped step
display(Image(filename=save_animation(render.animate(truth, dead, reckoned, closing), "odometry", 80)))

In [ ]:
# checks
# The gradient falls a millionfold from dead reckoning to the most likely poses.
at_dead = np.abs(gradient(dead).kernel).max()
np.testing.assert_allclose(gradient(poses).kernel, 0.0, atol=1e-6 * at_dead)